In [87]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [88]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
import numpy as np

In [89]:
sample_docs = ["""
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """]

In [90]:
import os



In [91]:
import tempfile
temp_dir = tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(os.path.join(temp_dir, f"doc_{i}.txt"), "w") as f:
        f.write(doc)

print(f"Documents saved in temporary directory: {temp_dir}")

Documents saved in temporary directory: /var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpalw0vf0y


In [92]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

In [93]:
loader = DirectoryLoader(temp_dir, loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'}, glob="*.txt")
documents = loader.load()
print(f"Loaded {len(documents)} documents.")

Loaded 3 documents.


In [94]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)

In [95]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2998.07it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [96]:
sample_text = """This is a sample text to demonstrate the functionality of the ChromaDB vector store.
It contains multiple sentences and is meant to be split into smaller chunks for processing."""



In [97]:
vector = embeddings.embed_query(sample_text)

In [98]:
vector

[-0.066373810172081,
 0.09631198644638062,
 -0.05204366520047188,
 -0.016056237742304802,
 0.01327076368033886,
 0.044619832187891006,
 0.005186716094613075,
 0.04167472943663597,
 -0.01683001033961773,
 -0.006033299025148153,
 0.0021720738150179386,
 -0.015062188729643822,
 0.027172477915883064,
 -0.07261878252029419,
 0.006117652170360088,
 0.07744006812572479,
 -0.07161134481430054,
 0.0028871078975498676,
 0.0403943732380867,
 -0.015122097916901112,
 -0.003919396083801985,
 -0.02623876929283142,
 -0.030766600742936134,
 -0.004169356543570757,
 -0.00784722063690424,
 0.1358095109462738,
 -0.09958479553461075,
 -0.011136187240481377,
 0.11938615143299103,
 -0.05070080608129501,
 0.04010456055402756,
 0.03943335637450218,
 0.14106053113937378,
 0.133452907204628,
 0.013897527009248734,
 0.05260603502392769,
 -0.04133649542927742,
 -0.02114935591816902,
 -0.07492396235466003,
 0.032059404999017715,
 0.031151771545410156,
 0.041743114590644836,
 -0.0639217346906662,
 0.07027482241392136

In [99]:
chunks

[Document(metadata={'source': '/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpalw0vf0y/doc_2.txt'}, page_content='Natural Language Processing (NLP)'),
 Document(metadata={'source': '/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpalw0vf0y/doc_2.txt'}, page_content='NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,'),
 Document(metadata={'source': '/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpalw0vf0y/doc_2.txt'}, page_content='machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand'),
 Document(metadata={'source': '/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpalw0vf0y/doc_2.txt'}, page_content='context and relationships between words in text.'),
 Document(metadata={'source': '/var/folders/ly/dps_wwpj797

In [100]:
persist_directory = "./chroma_db"

vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

print(f"Vector store created created with {vectorstore._collection.count()} vectors.")
print(f"Vector store persisted at: {persist_directory}")

Vector store created created with 72 vectors.
Vector store persisted at: ./chroma_db


In [101]:
query = "what is machine learning?"

similar_docs = vectorstore.similarity_search(query, k=4)

In [102]:
similar_docs

[Document(metadata={'source': '/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpwymf7lvr/doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main'),
 Document(metadata={'source': '/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpthos9avo/doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main'),
 Document(metadata={'source': '/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpk0i_dahw/doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main'),
 Document(metadata={'source': '/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpalw0vf0y/doc_0.tx

In [103]:
print(f'two most similar documents to the query "{query}":')
for i,doc in enumerate(similar_docs):
    print(f"\n--chunk {i+1}--")
    print(doc.page_content[:200]+"..,")
    print(f"metadata: {doc.metadata.get('source', 'N/A')}")


two most similar documents to the query "what is machine learning?":

--chunk 1--
Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main..,
metadata: /var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpwymf7lvr/doc_0.txt

--chunk 2--
Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main..,
metadata: /var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpthos9avo/doc_0.txt

--chunk 3--
Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main..,
metadata: /var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpk0i_dahw/doc_0.txt

--chunk 4--
Machine learning is a subset of artificial intelligence that enables systems to learn 
    and i

In [104]:
results_scores = vectorstore.similarity_search_with_score(query, k=4)

print(f'Two most similar documents to the query "{query}":')
for i, (doc, score) in enumerate(results_scores):
    print(f"\n--chunk {i+1}--")
    print(doc.page_content[:200] + "..,")
    print(f"metadata: {doc.metadata.get('source', 'N/A')}")
    print(f"score: {score}")


Two most similar documents to the query "what is machine learning?":

--chunk 1--
Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main..,
metadata: /var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpwymf7lvr/doc_0.txt
score: 0.3908127546310425

--chunk 2--
Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main..,
metadata: /var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpthos9avo/doc_0.txt
score: 0.3908127546310425

--chunk 3--
Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main..,
metadata: /var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpk0i_dahw/doc_0.txt
score: 0.3908127546310425

--chunk 4--
Machine learning i

In [105]:
import os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [106]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.7)

In [107]:
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate



In [108]:
retriever  = vectorstore.as_retriever(search_kwargs={"k": 4})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x138623210>, search_kwargs={'k': 4})

In [109]:
system_prompt = """You are a helpful assistant for answering questions based on retrieved documents.
Use only the following retrieved documents to answer the question. If you don't know the answer, say you don't know. Always use all the retrieved documents to answer the question.
Context:
{context}"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [110]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are a helpful assistant for answering questions based on retrieved documents.\nUse only the following retrieved documents to answer the question. If you don't know the answer, say you don't know. Always use all the retrieved documents to answer the question.\nContext:\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [111]:
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

In [112]:
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are a helpful assistant for answering questions based on retrieved documents.\nUse only the following retrieved documents to answer the question. If you don't know the answer, say you don't know. Always use all the retrieved documents to answer the question.\nContext:\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'ima

In [113]:
rag_chain = create_retrieval_chain(retriever,document_chain)

In [114]:
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x138623210>, search_kwargs={'k': 4}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are a helpful assistant for answering questions based on retrieved documents.\nUse only the following retrieved documents to ans

In [115]:
response = rag_chain.invoke({"input": "what is machine learning?"})

In [116]:
response

{'input': 'what is machine learning?',
 'context': [Document(metadata={'source': '/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpwymf7lvr/doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main'),
  Document(metadata={'source': '/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpthos9avo/doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main'),
  Document(metadata={'source': '/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/tmpk0i_dahw/doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main'),
  Document(metadata={'source': '/var/folders/ly